# Day 4 — Exercise: Adjusted Prices, Simple vs Log Returns

## Setup

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)
DATA_SOURCE = os.environ.get("QRC_DATA", "real")

from qrc.data import get_prices
from qrc.synth import synthetic_prices

## Part A — The dividend gap (the quiet 30% error)

**Real mode:** KO is a long-standing dividend payer. Pull 10+ years of BOTH
`adj_close` and `raw_close` (see `get_prices`' `field` parameter — read its
docstring first). **Synthetic mode:** the code below fabricates a stock with
a 3% annual dividend yield, paid smoothly.

In [ ]:
if DATA_SOURCE == "real":
    adj = get_prices("KO", start="2012-01-01", field="adj_close")
    raw = get_prices("KO", start="2012-01-01", field="raw_close")
    n_years = (adj.index[-1] - adj.index[0]).days / 365.25
else:
    rng = np.random.default_rng(11)
    n, p0, yld = 2500, 100.0, 0.03
    r_price = rng.normal(0.0002, 0.010, n)              # price-only returns
    idx = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=n)
    raw = pd.DataFrame({"KO": p0 * np.cumprod(1 + r_price)}, index=idx)
    adj = pd.DataFrame({"KO": (p0 * np.cumprod(1 + r_price + yld / 252))}, index=idx)
    n_years = n / 252

print(f"sample: {n_years:.1f} years")

Compute, over the full sample, the **annualized total return** (from
`adj`) and the **annualized price-only return** (from `raw`), and the gap in
percentage points per year.

In [ ]:
def annualized_return(price_series, n_years):
    """CAGR from a price series: (end/start) ** (1/years) - 1."""
    # YOUR CODE (hint: geometric mean, not arithmetic)
    ...

# total_cagr  = annualized_return(adj["KO"], n_years)
# price_cagr  = annualized_return(raw["KO"], n_years)
# print(f"total-return CAGR: {total_cagr:.2%} | price-only CAGR: {price_cagr:.2%}")
# print(f"gap: {(total_cagr - price_cagr) * 100:.2f} percentage points per year")

**A1.** Explain the gap: where does it come from, and what would using
`raw_close` do to any study of this stock's returns?

In [ ]:
# Your answer:

**A2.** Cumulative damage: plot the growth of $1 under both series over the
sample. By what factor do they differ at the end?

In [ ]:
# YOUR CODE

## Part B — Simple vs log returns, measured

Using `adj` (real mode) or `raw` (synthetic mode), compute daily simple
returns `r` and daily log returns `lr`.

**B1.** Verify over five random weeks: the sum of daily log returns within a
week equals (up to floating point) the log of the weekly growth factor,
$\ln(P_{fri}/P_{last fri})$. Show it for two of them.

In [ ]:
# YOUR CODE

**B2.** Make a scatter plot of `r` (x) vs `lr` (y) for all days, plus the
45° line. For what range of |r| do they differ by less than 10% *relative*
error? (Relative: |lr − r| / |r|.)

In [ ]:
# YOUR CODE — hint: np.log1p(r) is the log return

**B3.** Crash asymmetry: convert −10%, −50%, −90% simple returns to log
returns. Why will "log returns are symmetric" be a *modeling assumption*
rather than a fact?

In [ ]:
for r in [-0.10, -0.50, -0.90]:
    pass  # YOUR CODE: print simple and log side by side

## Part C — First research charts

Two-panel figure (level + daily returns) for your series, with the largest
single-day gain and loss marked (annotate the dates).

In [ ]:
# YOUR CODE

## Hints (ordered)

- A: CAGR uses the geometric mean — compound the whole sample, then take the
  1/n_years root.
- B1: `np.log1p(r).resample("W-FRI").sum()` vs `np.log(P_fri / P_last_fri)`.
- B2: near zero, ln(1+r) ≈ r − r²/2; the divergence grows quadratically.